<a href="https://colab.research.google.com/github/syoon7079/tjwn/blob/main/12_sensitivity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#11주차 노트북 준비

In [1]:
# =============================================================================
# [12주차 민감도 분석 노트북 시작부] 11주차 주 분석(모형 2) 재현 코드
# =============================================================================
!pip install pyreadstat -q

import os
import numpy as np
import pandas as pd
import pyreadstat
import statsmodels.api as sm
import statsmodels.formula.api as smf

from google.colab import drive
drive.mount('/content/drive')

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 200)

DATA_DIR = '/content/drive/MyDrive/hdac_data'
SAV_PATH = f'{DATA_DIR}/HN24_ALL.sav'
OUT_DIR  = f'{DATA_DIR}/output'
os.makedirs(OUT_DIR, exist_ok=True)

AGE_MIN = 19
WEIGHT = 'wt_tot'

# 1. 11주차 필수 변수 로드
NEEDED = ['year', 'l_out_fq', 'he_bmi', 'bd1_11', 'region', 'sex', 'age', 'educ', 'incm', 'bs3_1',
          'kstrata', 'psu', WEIGHT]

_, m0 = pyreadstat.read_sav(SAV_PATH, metadataonly=True)
low2real = {c.lower(): c for c in m0.column_names}
df, meta = pyreadstat.read_sav(SAV_PATH, usecols=[low2real[c] for c in NEEDED])
df.columns = df.columns.str.lower()

# 2. 11주차 변수 전처리 및 grp(외식 빈도 3분류) 생성
def num(s): return pd.to_numeric(s, errors='coerce')

EXP_VALID = [1, 2, 3, 4, 5, 6, 7]
GRP_MAP   = {5: 1, 6: 1, 7: 1, 4: 2, 1: 3, 2: 3, 3: 3} # 1: 고빈도, 2: 중빈도, 3: 저빈도
df['exp_raw']   = num(df['l_out_fq'])
df['exp_valid'] = df['exp_raw'].isin(EXP_VALID)
df['grp']       = df['exp_raw'].where(df['exp_valid']).map(GRP_MAP)

df['bmi']     = num(df['he_bmi'])
df['age_n']   = num(df['age'])
df['w']       = num(df[WEIGHT])

# 공변량 매핑
df['c_sex']   = num(df['sex']).map({1: '남성', 2: '여성'})
df['c_educ']  = num(df['educ']).mask(num(df['educ']).isin([88, 99]))
df['c_incm']  = num(df['incm']).mask(num(df['incm']).isin([8, 9, 99]))
df['c_bd1_11']= num(df['bd1_11']).mask(num(df['bd1_11']).isin([9, 99]))
df['c_bs3_1'] = num(df['bs3_1']).mask(num(df['bs3_1']).isin([9, 99]))

# 3. 조사 설계 및 분석 대상자(analytic) 선정
psu_str = df['psu'].astype(str).str.strip()
df['psu_ok']    = df['psu'].notna() & ~psu_str.isin(['', 'nan', 'None'])
df['design_ok'] = df['w'].notna() & (df['w'] > 0) & df['kstrata'].notna() & df['psu_ok']

df['s1'] = True
df['s2'] = df['age_n'] >= AGE_MIN
df['s3'] = df['s2'] & df['exp_valid']
df['s4'] = df['s3'] & df['bmi'].notna()
df['s5'] = df['s4'] & df['design_ok']
df['analytic'] = df['s5']

# 4. 공변량 완전 사례(Complete Case) 필터링 (Table 1 대비 28명 차이인 5,737명 도출)
adjustment_cols = ['sex', 'region', 'educ', 'incm', 'bd1_11', 'bs3_1', 'age_n']
valid_mask = (
    df['analytic'] &
    df['grp'].notna() &
    df['bmi'].notna() &
    df[adjustment_cols].notna().all(axis=1) &
    df['design_ok']
)
sub_df = df[valid_mask].copy()
sub_df['grp'] = pd.Categorical(sub_df['grp'], categories=[3, 2, 1], ordered=True)

print(f"최종 완전 사례 분석 인원 (N): {len(sub_df):,}명 (11주차와 동일)")

# 5. 복합표본 가중 선형회귀 분석 함수 정의 및 모형 2 실행
def fit_wls(formula, data):
    model = smf.wls(formula, data=data, weights=data['w'])
    return model.fit(cov_type='cluster', cov_kwds={'groups': data['psu']})

# 모형 2 (다변량 보정) 적합
res_m2 = fit_wls('bmi ~ C(grp, Treatment(reference=3)) + C(sex) + C(region) + C(educ) + C(incm) + C(bd1_11) + C(bs3_1) + age_n', sub_df)

# 결과 출력 확인
from IPython.display import display, HTML

def format_result(res, param_key):
    if param_key == 'ref':
        return "Ref. (-)"

    target_name = [name for name in res.params.index if param_key in name][0]

    coef = res.params[target_name]
    se = res.bse[target_name]
    ci_low, ci_high = res.conf_int().loc[target_name]
    pval = res.pvalues[target_name]
    pval_str = f"{pval:.3f}" if pval >= 0.001 else "< 0.001"

    return f"β = {coef:.3f} (SE = {se:.3f})<br>95% CI: [{ci_low:.3f}, {ci_high:.3f}]<br>p-value = {pval_str}"

check_df = pd.DataFrame([
    {
        "구분": "12주차 재현 결과 (모형 2)",
        "분석 인원 (N)": f"{len(sub_df):,}명",
        "저빈도 외식군 (주 1-2회 이하)": format_result(res_m2, 'ref'),
        "중빈도 외식군 (주 3-4회)": format_result(res_m2, '[T.2]'),
        "고빈도 외식군 (주 5회 이상)": format_result(res_m2, '[T.1]')
    }
])

print("=== [12주차 노트북: 모형 2 재현 확인 표] ===")
display(HTML(check_df.to_html(index=False, escape=False)))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
최종 완전 사례 분석 인원 (N): 5,737명 (11주차와 동일)
=== [12주차 노트북: 모형 2 재현 확인 표] ===


구분,분석 인원 (N),저빈도 외식군 (주 1-2회 이하),중빈도 외식군 (주 3-4회),고빈도 외식군 (주 5회 이상)
12주차 재현 결과 (모형 2),"5,737명",Ref. (-),"β = -0.300 (SE = 0.207)95% CI: [-0.705, 0.105]p-value = 0.147","β = -0.489 (SE = 0.149)95% CI: [-0.782, -0.196]p-value = 0.001"


#D2 Table3

In [2]:
# ==============================================================================
# [D-2] Table 3: 주 분석 및 민감도 분석(S3, S4) 비교 출력 코드
# ==============================================================================
from IPython.display import display, HTML

# 1. S3를 위해 신체활동(pa_aerobic) 변수 로드 및 병합
# (앞선 셀에서 pa_aerobic을 불러오지 않았을 경우를 대비해 원자료에서 해당 컬럼만 안전하게 가져옴)
if 'pa_aerobic' not in df.columns:
    pa_col = low2real['pa_aerobic']
    df_pa, _ = pyreadstat.read_sav(SAV_PATH, usecols=[pa_col])
    df['pa_aerobic'] = num(df_pa[pa_col])
else:
    df['pa_aerobic'] = num(df['pa_aerobic'])

# 8(비해당), 9(모름/무응답) 결측 처리 (1: 실천, 0: 비실천)
df['c_pa'] = df['pa_aerobic'].mask(df['pa_aerobic'].isin([8, 9, 99]))

# 주 분석 표본(sub_df, N=5,737)에 c_pa 매핑
sub_df['c_pa'] = df.loc[sub_df.index, 'c_pa']

# ------------------------------------------------------------------------------
# 2. 모형 적합 (주 분석 표본 sub_df 기준)
# ------------------------------------------------------------------------------
# [주 분석] 모형 2 (11주차 동일)
formula_main = 'bmi ~ C(grp, Treatment(reference=3)) + C(sex) + C(region) + C(educ) + C(incm) + C(bd1_11) + C(bs3_1) + age_n'
res_main = fit_wls(formula_main, sub_df)

# [민감도 분석 S3] 신체활동(c_pa) 추가 보정
# 주 분석과 동일한 사람(sub_df)으로 적합하기 위해 결측이 있는 경우 제외하거나 동일 표본 확인
sub_df_s3 = sub_df.dropna(subset=['c_pa']).copy()
formula_s3 = formula_main + ' + C(c_pa)'
res_s3 = fit_wls(formula_s3, sub_df_s3)

# [민감도 분석 S4] 외식 빈도 원 코드(1~7) 연속형 투입 (주 분석과 완전히 동일한 5,737명)
formula_s4 = 'bmi ~ exp_raw + C(sex) + C(region) + C(educ) + C(incm) + C(bd1_11) + C(bs3_1) + age_n'
res_s4 = fit_wls(formula_s4, sub_df)

# ------------------------------------------------------------------------------
# 3. 주 분석과의 차이 계산 및 표 포맷팅 함수
# ------------------------------------------------------------------------------
main_b2 = res_main.params['C(grp, Treatment(reference=3))[T.2]']
main_b1 = res_main.params['C(grp, Treatment(reference=3))[T.1]']

def get_stats_str(res, param_name, ref_beta=None):
    b = res.params[param_name]
    se = res.bse[param_name]
    ci_l, ci_h = res.conf_int().loc[param_name]
    p = res.pvalues[param_name]
    p_str = f"{p:.3f}" if p >= 0.001 else "< 0.001"

    stat_txt = f"β = {b:.3f} (SE = {se:.3f})<br>95% CI: [{ci_l:.3f}, {ci_h:.3f}]<br>p = {p_str}"

    if ref_beta is None:
        diff_txt = "기준 (Main)"
    else:
        diff = b - ref_beta
        pct = (diff / abs(ref_beta)) * 100
        diff_txt = f"Δβ = {diff:+.3f} ({pct:+.1f}%)"
    return stat_txt, diff_txt

# 각 모형별 수치 추출
m_mid, m_mid_d   = get_stats_str(res_main, 'C(grp, Treatment(reference=3))[T.2]', None)
m_high, m_high_d = get_stats_str(res_main, 'C(grp, Treatment(reference=3))[T.1]', None)

s3_mid, s3_mid_d   = get_stats_str(res_s3, 'C(grp, Treatment(reference=3))[T.2]', main_b2)
s3_high, s3_high_d = get_stats_str(res_s3, 'C(grp, Treatment(reference=3))[T.1]', main_b1)

s4_cont, _ = get_stats_str(res_s4, 'exp_raw', None)

# ------------------------------------------------------------------------------
# 4. Table 3 데이터프레임 구성 및 출력
# ------------------------------------------------------------------------------
table3_data = [
    {
        "분석 구분": "<b>주 분석 (Main, 모형 2)</b>",
        "분석 인원 (N)": f"{len(sub_df):,}명",
        "중빈도 외식군 (주 3-4회)<br>계수 · 95% CI · p값": m_mid,
        "고빈도 외식군 (주 5회 이상)<br>계수 · 95% CI · p값": m_high,
        "주 분석과의 차이 (Δβ)": "기준 모형 (Ref.)"
    },
    {
        "분석 구분": "<b>S3. 신체활동 추가 보정</b><br>(pa_aerobic 투입)",
        "분석 인원 (N)": f"{len(sub_df_s3):,}명",
        "중빈도 외식군 (주 3-4회)<br>계수 · 95% CI · p값": s3_mid,
        "고빈도 외식군 (주 5회 이상)<br>계수 · 95% CI · p값": s3_high,
        "주 분석과의 차이 (Δβ)": f"중빈도: {s3_mid_d}<br>고빈도: {s3_high_d}"
    },
    {
        "분석 구분": "<b>S4. 노출 정의 변경 (연속형)</b><br>(원 코드 1~7점 투입)",
        "분석 인원 (N)": f"{len(sub_df):,}명",
        "중빈도 외식군 (주 3-4회)<br>계수 · 95% CI · p값": f"[연속형 1단위 증가당]<br>{s4_cont}",
        "고빈도 외식군 (주 5회 이상)<br>계수 · 95% CI · p값": "상동 (단일 선형 기울기)",
        "주 분석과의 차이 (Δβ)": "음(-)의 방향성 및 통계적 유의성 일치"
    }
]

df_table3 = pd.DataFrame(table3_data)

print("=== [D-2 Table 3] 민감도 분석 결과 비교표 ===")
display(HTML(df_table3.to_html(index=False, escape=False)))

# S3가 동일 표본(5,737명)에서 결측이 있었는지 안내 출력
if len(sub_df) != len(sub_df_s3):
    print(f"\n※ 참고: 주 분석 대상자 {len(sub_df):,}명 중 신체활동(pa_aerobic) 결측 {len(sub_df)-len(sub_df_s3)}명이 제외되어 S3는 {len(sub_df_s3):,}명으로 적합됨.")

=== [D-2 Table 3] 민감도 분석 결과 비교표 ===


분석 구분,분석 인원 (N),중빈도 외식군 (주 3-4회)계수 · 95% CI · p값,고빈도 외식군 (주 5회 이상)계수 · 95% CI · p값,주 분석과의 차이 (Δβ)
"주 분석 (Main, 모형 2)","5,737명","β = -0.300 (SE = 0.207)95% CI: [-0.705, 0.105]p = 0.147","β = -0.489 (SE = 0.149)95% CI: [-0.782, -0.196]p = 0.001",기준 모형 (Ref.)
S3. 신체활동 추가 보정(pa_aerobic 투입),"5,340명","β = -0.242 (SE = 0.204)95% CI: [-0.643, 0.158]p = 0.236","β = -0.478 (SE = 0.151)95% CI: [-0.773, -0.183]p = 0.002",중빈도: Δβ = +0.058 (+19.3%)고빈도: Δβ = +0.012 (+2.4%)
S4. 노출 정의 변경 (연속형)(원 코드 1~7점 투입),"5,737명","[연속형 1단위 증가당]β = -0.154 (SE = 0.045)95% CI: [-0.241, -0.067]p = < 0.001",상동 (단일 선형 기울기),음(-)의 방향성 및 통계적 유의성 일치



※ 참고: 주 분석 대상자 5,737명 중 신체활동(pa_aerobic) 결측 397명이 제외되어 S3는 5,340명으로 적합됨.
